# 28 · Applications créatives — solutions (notebook exécuté)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/chapitres/ch28_creatif/05_solutions.ipynb)

Les réponses des exercices, exécutées. Les démarches détaillées (le *pourquoi*, les erreurs fréquentes, les variantes) sont dans `05_solutions.md`. Les cellules marquées `answer` enregistrent les réponses vérifiées par `wb.check` (`tools/build_answers.py`).

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | 28.Q1 à 28.Q10, 28.R1 à 28.R3, 28.1, 28.2, 28.4 | vérifier tes réponses courtes | 🧠 🔁 ✏️ | ★ à ★★ | |
| A | 28.11 à 28.15 | visualiser filtres et couches : un extracteur par hooks, la montée de gradient sur un réseau ImageNet, une galerie, la régularisation (*prochaine session de génération*) | | | |
| B | 28.16 et 28.17 | le deep dream sur une photo libre, couches, poids et masques (*prochaine session de génération*) | | | |
| C | 28.18 à 28.26 | le transfert de style : la matrice de Gram, des courbes, un 🐛, les reconstructions, le transfert complet, ses réglages, au-delà de Gram, une galerie documentée (*prochaine session de génération*) | | | |
| D | 28.27 à 28.29 | le texte mot à mot sur Verne, les stratégies de décodage, et le défi (*prochaine session de génération*) | | | |

In [1]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mylearn = wb.load_mylearn("ref")  # reference implementation

🧪 Deep Learning Workbook : environnement
   Plateforme : Local (Linux x86_64, 2 CPU)
   Python 3.13.18 | numpy 2.1.3 | pandas 2.2.3 | matplotlib 3.10.0 | scikit-learn 1.6.1 | scipy 1.16.3 | torch 2.11.0+cpu | torchvision 0.26.0+cpu
   Device : cpu | Graine : 42 | FAST_MODE : True


## Objectifs et rappel express

- Extraire les activations d'un réseau pré-entraîné gelé avec des hooks, et visualiser ce que cherchent ses filtres et ses couches par montée de gradient sur l'image.
- Faire rêver un réseau sur une photo libre (deep dream).
- Écrire la matrice de Gram et les losses de contenu, de style et de variation totale, puis un transfert de style complet sous budget CPU, et documenter une galerie (sources, licences, réglages).
- Générer du texte mot à mot sur Verne et comparer les stratégies de décodage.

**Rappel express.** Montée de gradient : $\mathbf{x} \leftarrow \mathbf{x} + \eta\, \nabla_{\mathbf{x}}\, a(\mathbf{x})$, poids gelés. Gram : $G = \frac{1}{CHW} F F^\top$. Transfert de style : $L = \alpha \lVert F(\mathbf{x}) - F(\mathbf{c}) \rVert^2 + \beta \sum_\ell w_\ell \lVert G^\ell(\mathbf{x}) - G^\ell(\mathbf{s}) \rVert_F^2 + \lambda L_{TV}$. Température : $p_i \propto e^{z_i / T}$.

## Partie 0 · Vérifier tes réponses courtes (quiz, rappels, ✏️ 28.1, 28.2 et 28.4)

Fais d'abord les quiz, les rappels et les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse courte ici : **la valeur** que tu as trouvée, pas l'expression Python, sinon tu ne vérifies rien. Un nombre : `42` ou `0.3` (en Python, le séparateur décimal est un **point** ; `0,3` sans guillemets serait un couple de deux nombres) ; un vrai ou faux : `True` ou `False` ; un choix : la lettre seule, entre guillemets (`"B"`) ; plusieurs lettres : entre guillemets (`"ABD"`). Arrondis comme l'énoncé le demande, et seulement à la fin du calcul. Les réponses pas encore remplies affichent ⏳. Les questions « dans ta copie », les dérivations ∂ 28.3 et 28.5, les estimations 🧮 28.6 et 28.7, la réflexion (🗣️ 28.8, ⚖️ 28.9, 📄 28.10) et l'entretien se corrigent avec `05_solutions.md`.

In [2]:
# The short answers only use the numbers of their statements (02_exercices.md)
import numpy as np

# VGG16: (convolutions, filters) of the five blocks; a 2x2 max-pooling of stride 2 ends each block
VGG16_BLOCKS = [(2, 64), (2, 128), (3, 256), (3, 512), (3, 512)]


def side(block, image=224):
    """Side of the activation maps of the convolutions of a block (1 to 5)."""
    return image // 2 ** (block - 1)


F_28_2 = np.array([[1, 0, 2, 1], [0, 3, 1, 1], [2, 0, 4, 2]], dtype=float)
RAW_28_2 = F_28_2 @ F_28_2.T
G_28_2 = RAW_28_2 / F_28_2.size

FX_28_4, FC_28_4 = np.array([1, 2, 0, 1.0]), np.array([1, 0, 1, 1.0])
SX_28_4, SS_28_4 = np.array([[1, 1], [0, 2.0]]), np.array([[2, 0], [0, 2.0]])
GX_28_4, GS_28_4 = SX_28_4 @ SX_28_4.T / SX_28_4.size, SS_28_4 @ SS_28_4.T / SS_28_4.size
IMG_28_4 = np.array([[0, 1, 0], [1, 1, 1], [0, 1, 3.0]])
LC_28_4 = float(((FX_28_4 - FC_28_4) ** 2).sum())
LS_28_4 = float(((GX_28_4 - GS_28_4) ** 2).sum())
LTV_28_4 = float(((IMG_28_4[1:] - IMG_28_4[:-1]) ** 2).sum() + ((IMG_28_4[:, 1:] - IMG_28_4[:, :-1]) ** 2).sum())

**28.Q1 — Pourquoi partir d'un réseau pré-entraîné comme VGG16**

In [3]:
wb.record("28.Q1a", False, mistakes={"relis le §28.2.2 : que modifie-t-on à chaque pas, et que garde-t-on fixe ?": True})
wb.record("28.Q1b", 13 + 3, mistakes={"les 13 convolutions ne sont pas tout : ajoute les couches denses": 13,
          "le max-pooling n'a pas de poids : ne compte que les convolutions et les couches denses": 21})
wb.record("28.Q1c", "B", mistakes={"VGG16 est un gros réseau, lent pour sa précision : ce n'est pas la raison": "A", "ImageNet contient des photos d'objets et d'animaux, pas des tableaux": "C"})
wb.record("28.Q1d", "B", mistakes={"c'est le chiffre du livre, corrigé par la fiche (§28.2.1)": "A", "14 millions est la taille de la base ImageNet complète, pas du jeu d'entraînement d'ILSVRC": "C"})
wb.record("28.Q1e", 1000, mistakes={"ImageNet-1k : le nom donne le nombre de classes": 21000})

WB_ANSWER {"hash": "7034d074a093cec6238ef634513f0c5c01eb66259e1300cf70e3a7a16e69ea49", "id": "28.Q1a", "kind": "bool", "mistakes": {"9c1486e18c68c09a642d03639a905b1f511ec65e5d8d2e6b39f740b00c8b2a8d": "relis le §28.2.2 : que modifie-t-on à chaque pas, et que garde-t-on fixe ?"}}
📝 Ex 28.Q1a : réponse enregistrée (bool).
WB_ANSWER {"hash": "1eb408fe4395f378d64a4e68372d1aa989ae767394b09aab23933477549814e8", "id": "28.Q1b", "kind": "int", "magnitude": 1, "mistakes": {"42c573ae6ca792fd7f449690ed66d96e4719554ca4cc5ee011e11fdd03283871": "le max-pooling n'a pas de poids : ne compte que les convolutions et les couches denses", "8512dc3f15606c96de7b66fa8ea21dec1189383ba555d9b94dac0f557065d927": "les 13 convolutions ne sont pas tout : ajoute les couches denses"}, "sign": 1}
📝 Ex 28.Q1b : réponse enregistrée (int).
WB_ANSWER {"hash": "d2a7546a2e7c829f8ce579d7d3d0abb4ec27bed97d0ba1ff92f9555f706934c7", "id": "28.Q1c", "kind": "str", "mistakes": {"19d835c9aec190e40a83013988e6c461205a6d967c24299e24a1e

**28.Q2 — Montée de gradient sur l'entrée, poids gelés**

In [4]:
wb.record("28.Q2a", "B", mistakes={"les poids du réseau restent fixes pendant toute la visualisation": "A", "le learning rate est un réglage, pas ce qu'on optimise": "C"})
wb.record("28.Q2b", True, mistakes={"monter sur a, c'est descendre sur −a : le gradient change de signe, le pas aussi": False})
wb.record("28.Q2c", "A", mistakes={"c'est le deep dream qui part d'une photo, pas la visualisation d'un filtre": "B", "relis le §28.2.2 : des valeurs tirées au hasard entre −1 et 1": "C"})
wb.record("28.Q2d", "B", mistakes={"l'optimiseur ne reçoit que ce qu'on veut modifier : l'image": "A"})
wb.record("28.Q2e", True, mistakes={"geler les poids coupe leur gradient, pas celui de l'image, qui a requires_grad=True": False})

WB_ANSWER {"hash": "9197b1913558fb5e1c2b28db5c643c96809944c32b36a163a6a2d4098e36f51e", "id": "28.Q2a", "kind": "str", "mistakes": {"bbf55a165a1f0721e555f3b25c1264d69c1e6c90791c00be1f5bf0a531974bd6": "le learning rate est un réglage, pas ce qu'on optimise", "d8dd2e7d7f96dcb9d0aab8fd750f49b268878fc4e37d8f15d03bd55753d73556": "les poids du réseau restent fixes pendant toute la visualisation"}}
📝 Ex 28.Q2a : réponse enregistrée (str).
WB_ANSWER {"hash": "2fe3bf971f93e92728564bb08f6f2f46a2eff17e7a3c10ea0ba0a54c1ce2c982", "id": "28.Q2b", "kind": "bool", "mistakes": {"9b5a4c4a5251c0fc7b93cdc81ff02443ff5cbb42a3bf38e5868afa3d3d552fdc": "monter sur a, c'est descendre sur −a : le gradient change de signe, le pas aussi"}}
📝 Ex 28.Q2b : réponse enregistrée (bool).
WB_ANSWER {"hash": "7c321cc355c7fb3fdb3af448b394fd38e96fec750d550a458ad9d2cb88f5d500", "id": "28.Q2c", "kind": "str", "mistakes": {"4152ef56eae0ab47f591196d221a58fea00ff1df949b63d6b8a3555b03b9250c": "relis le §28.2.2 : des valeurs tirées 

**28.Q3 — Visualiser une couche entière, pondérer ses filtres**

In [5]:
wb.record("28.Q3a", "B", mistakes={"un seul filtre, c'est la visualisation d'un filtre (§28.2.2)": "A", "les couches denses ne servent pas ici : on regarde les convolutions": "C"})
wb.record("28.Q3b", "A", mistakes={"les parties d'objets apparaissent dans les couches profondes, pas au bloc 1": "B"})
wb.record("28.Q3c", True, mistakes={"avec un poids négatif, augmenter la somme pondérée, c'est diminuer l'activation de ce filtre": False})
wb.record("28.Q3d", 512, mistakes={"les filtres doublent de bloc en bloc jusqu'au bloc 4, puis restent à 512": 256})
wb.record("28.Q3e", side(4), mistakes={"le bloc 4 vient après trois poolings, pas quatre": 14,
          "le bloc 4 vient après trois poolings, et chacun divise le côté par 2": 56})

WB_ANSWER {"hash": "9e8248f218f742377441f14c013a47a7fa10d2db64a6c167143a27ea8365d6db", "id": "28.Q3a", "kind": "str", "mistakes": {"5003f49390d8f82c81683ff770596e09c5658795052d9fe0d3f9280c59b77cc9": "un seul filtre, c'est la visualisation d'un filtre (§28.2.2)", "7b9fcfa6b5a2679506ddd5f0fa60053ce29e1c2c0f56be999e2494ff9fd4d401": "les couches denses ne servent pas ici : on regarde les convolutions"}}
📝 Ex 28.Q3a : réponse enregistrée (str).
WB_ANSWER {"hash": "ecb470e9f953c5f6709d1de59bb3fc25b7e7bf148c03d819574557c59ed68fb0", "id": "28.Q3b", "kind": "str", "mistakes": {"870734d2711506ef3fd67f311db2ec840c5c5bf66de0d455c0f1d717e0f154ab": "les parties d'objets apparaissent dans les couches profondes, pas au bloc 1"}}
📝 Ex 28.Q3b : réponse enregistrée (str).
WB_ANSWER {"hash": "c17ed6679dc06146ae2ef2b6996b130820495db29f210e37dfeae812f974b687", "id": "28.Q3c", "kind": "bool", "mistakes": {"a33a5da556043a092de3b29b846df442da6154b0046e17c0713d052c9756f392": "avec un poids négatif, augmenter la

**28.Q4 — Deep dream : ce qui change par rapport à la visualisation d'une couche**

In [6]:
wb.record("28.Q4a", "AB", mistakes={"les poids du réseau ne changent jamais dans ce chapitre": "ABC",
          "le deep dream fait monter les activations, comme la visualisation": "ABD",
          "il manque le changement d'image de départ": "B",
          "il manque la combinaison de plusieurs couches": "A"})
wb.record("28.Q4b", True, mistakes={"la montée de gradient renforce ce qui excite déjà les filtres choisis": False})
wb.record("28.Q4c", "A", mistakes={"relis le §28.3 : le nom du billet vient du réseau de Google qu'il utilisait": "B", "relis le §28.3 : le billet porte le nom d'un réseau de Google, pas d'une technique de peinture": "C"})
wb.record("28.Q4d", "C", mistakes={"couper le réseau change les couches, pas la zone de l'image": "A", "un pas plus petit ralentit le rêve partout, pas seulement dans une zone": "B"})
wb.record("28.Q4e", False, mistakes={"le transfert de style cherche à ressembler à des cibles : il fait baisser des écarts": True})

WB_ANSWER {"hash": "05c5233c246d40fc6bf3f5661042da4adcccf1677bfeb3cb6a6398029b751626", "id": "28.Q4a", "kind": "str", "mistakes": {"1a3e9034d54b7808a439628b39205ff83d0104edd966ebc16d29f894b83698ce": "le deep dream fait monter les activations, comme la visualisation", "24fb8bc5667d52efe73e8f2981646f3a3207e04c28b8cd855ee2921d3b087f51": "il manque la combinaison de plusieurs couches", "54b4347932a534e9c19e197b694b8861b934a26e8587273eaf3998874f61a232": "les poids du réseau ne changent jamais dans ce chapitre", "d258cc819d18fc47694f0376c6ec3f54eb2d73a7e99269a60fdeb1d1bc98f04f": "il manque le changement d'image de départ"}}
📝 Ex 28.Q4a : réponse enregistrée (str).
WB_ANSWER {"hash": "8396f170a8cf40b2d13bbd1820febd2e7b69641dddb951e6d720aa6777bafa66", "id": "28.Q4b", "kind": "bool", "mistakes": {"5b71b6fb0387fe450e1469b1e20b36e75fa9e9c58fce6681224b05675219a278": "la montée de gradient renforce ce qui excite déjà les filtres choisis"}}
📝 Ex 28.Q4b : réponse enregistrée (bool).
WB_ANSWER {"hash"

**28.Q5 — Des yeux partout : l'écho des données d'entraînement**

In [7]:
wb.record("28.Q5a", "C", mistakes={"aucune couche n'est spécialisée à l'avance : les détecteurs sont appris": "A", "le deep dream part d'une photo, pas d'un bruit": "B"})
wb.record("28.Q5b", False, mistakes={"plus d'une centaine des 1 000 classes sont des races de chiens": True})
wb.record("28.Q5c", "B", mistakes={"un réseau ne rêve que de ce qu'il a appris": "A"})
wb.record("28.Q5d", "B", mistakes={"les traits et les textures, ce sont les couches précoces": "A"})
wb.record("28.Q5e", True, mistakes={"ce que le réseau fait apparaître dépend de ses données d'entraînement": False})

WB_ANSWER {"hash": "2dce2bbe1695c606fb86f924a04f65d12c3337e580499e86703c2869b389911e", "id": "28.Q5a", "kind": "str", "mistakes": {"14ba0bb311222045d2118041fc64fa946ddb30ae5c306a8a5c8bbab6006b5e03": "le deep dream part d'une photo, pas d'un bruit", "da69356fc7c53f27337fd48f0d2c03c78e9c6f34cbfd5947b43609c53e328818": "aucune couche n'est spécialisée à l'avance : les détecteurs sont appris"}}
📝 Ex 28.Q5a : réponse enregistrée (str).
WB_ANSWER {"hash": "03b1b8e310706fb5901b7340f1fc4a6acc2cc4752ee7b2c5a7462f0b899b5fa1", "id": "28.Q5b", "kind": "bool", "mistakes": {"8c9eee895f62901ce013679ea8e96215c008aa420e730af3ef4363ccf3eb406a": "plus d'une centaine des 1 000 classes sont des races de chiens"}}
📝 Ex 28.Q5b : réponse enregistrée (bool).
WB_ANSWER {"hash": "8bc514568452c8e030ed7b00661e739a17554f5e39f1eca82bc1e18033f1c8f0", "id": "28.Q5c", "kind": "str", "mistakes": {"5eb672ce9d20cf9bdb1e38ddda5653db34cb490d5659f2eef69bfd99ee6ed040": "un réseau ne rêve que de ce qu'il a appris"}}
📝 Ex 28.Q5c

**28.Q6 — Image de base, référence de style, image générée : que minimise-t-on ?**

In [8]:
wb.record("28.Q6a", "C", mistakes={"la photo de contenu est une cible, elle ne change pas": "A", "le tableau est une cible, il ne change pas": "B", "le réseau est gelé pendant tout le transfert": "D"})
wb.record("28.Q6b", "A", mistakes={"le style se compare par les matrices de Gram, le contenu par les activations": "B"})
wb.record("28.Q6c", "B", mistakes={"la photo fournit le contenu, pas le style": "A"})
wb.record("28.Q6d", False, mistakes={"le réseau sert d'instrument de mesure : ses poids ne bougent pas": True})
wb.record("28.Q6e", 1, mistakes={"les cibles sont déjà calculées : seule l'image générée change d'un pas à l'autre": 3,
          "les cibles sont calculées une fois : seule l'image générée repasse": 2})

WB_ANSWER {"hash": "0716e5a88e4281da40af8bf913e818781f14ea421bcb0e1a2b8ce16b0078fd75", "id": "28.Q6a", "kind": "str", "mistakes": {"43145bf9f043e144090257f739c979d8466a6d43f34626d385f1c73296f7787e": "le réseau est gelé pendant tout le transfert", "70c164926bc30668c33aac03c2ca518b4625851355cb637ee08ff9d1fa562b7f": "la photo de contenu est une cible, elle ne change pas", "b6bed5a8d206df8e7895ded5ab6201a66af6d19f9508871d4354f0abe52cc206": "le tableau est une cible, il ne change pas"}}
📝 Ex 28.Q6a : réponse enregistrée (str).
WB_ANSWER {"hash": "92ade09d938939bdb16d889c3daa17fd8956a91f51e3ea1bda4fe5404c2d9c89", "id": "28.Q6b", "kind": "str", "mistakes": {"303048c966739256c59e5d216d8a13f6a71e979f64eb84c6d0ea50c2c9d03f22": "le style se compare par les matrices de Gram, le contenu par les activations"}}
📝 Ex 28.Q6b : réponse enregistrée (str).
WB_ANSWER {"hash": "bacb5aceaf271308126bb96d71c8acd67698b1a584cdd663a3db992fb862c613", "id": "28.Q6c", "kind": "str", "mistakes": {"18f11597dcf96541df8

**28.Q7 — Ce que la matrice de Gram retient et ce qu'elle oublie**

In [9]:
wb.record("28.Q7a", "A", mistakes={"on compare des filtres deux à deux, pas des positions": "B", "c'est la forme de F, la matrice des cartes mises à plat": "C"})
wb.record("28.Q7b", "B", mistakes={"les indices de G sont des filtres, pas des pixels": "A", "la somme porte sur toutes les positions : la position est oubliée": "C"})
wb.record("28.Q7c", True, mistakes={"chaque coefficient est une somme sur les positions : l'ordre des termes ne compte pas": False})
wb.record("28.Q7d", "B", mistakes={"G_ii additionne des carrés d'activations, pas des présences": "A", "G_ii additionne sur toutes les positions, pas seulement la plus forte": "C"})
wb.record("28.Q7e", False, mistakes={"relis l'encadré ⚠️ « personne ne sait vraiment » du §28.4.6 : Li et coll. nomment cette mesure": True})

WB_ANSWER {"hash": "f2d66f28cb12fdae7ef1b91d5935e884066bf25bedf35a4e182e87a90269aa95", "id": "28.Q7a", "kind": "str", "mistakes": {"5e746db08de8ebedb6d5d676251443154b35977699d3581e00ea8fa7edb0c741": "c'est la forme de F, la matrice des cartes mises à plat", "e74a2e35bb986b63234588b93f453d8596828fce78f905668524a0295f98b4ec": "on compare des filtres deux à deux, pas des positions"}}
📝 Ex 28.Q7a : réponse enregistrée (str).
WB_ANSWER {"hash": "68bc15e0a787ab26f914fee31bf8f69f2ac3e5bfad90907c2843c83c670fd1cd", "id": "28.Q7b", "kind": "str", "mistakes": {"2e582cee91d3e86667f2fbd95fe0adb9e0fccde486b9fc0a66187eb549ef9c9d": "la somme porte sur toutes les positions : la position est oubliée", "7ab74c22a5c6b9bba3deb238068e05b9c26e62a575960d0da846ac2bb1c349e3": "les indices de G sont des filtres, pas des pixels"}}
📝 Ex 28.Q7b : réponse enregistrée (str).
WB_ANSWER {"hash": "7c55b50f9161e7409fd9aae8b283e10c0a91f9509ca41e407f738a0b362e35d6", "id": "28.Q7c", "kind": "bool", "mistakes": {"2e340cb3f39

**28.Q8 — Loss de contenu : couches précoces ou profondes ?**

In [10]:
wb.record("28.Q8a", "A", mistakes={"les couches précoces gardent les détails : la reconstruction est presque exacte": "B"})
wb.record("28.Q8b", "B", mistakes={"plus on va profond, plus la reconstruction s'éloigne des pixels": "A"})
wb.record("28.Q8c", "C", mistakes={"c'est le choix du code de Chollet, pas celui du livre": "A", "toutes les couches, c'est le choix du livre pour le style": "B"})
wb.record("28.Q8d", False, mistakes={"Gatys et coll. prennent conv4_2, une couche profonde (fiche §28.4.3)": True})
wb.record("28.Q8e", "B", mistakes={"le nombre de paramètres n'a rien à voir avec ce que la loss impose": "A", "le temps de calcul n'a rien à voir avec ce que la loss impose": "C"})

WB_ANSWER {"hash": "ecf308271af2f327f9c17f74b2cee9732650f286d6553c8712b69e92f791421c", "id": "28.Q8a", "kind": "str", "mistakes": {"349645cc4e375617e638608c19f8f2d285e5cc999b335fef3abd7121bb422232": "les couches précoces gardent les détails : la reconstruction est presque exacte"}}
📝 Ex 28.Q8a : réponse enregistrée (str).
WB_ANSWER {"hash": "803d1c3e0fd3878b2dee9e9b1645ea00e7ba1f0cf224b0a030b62e097d222749", "id": "28.Q8b", "kind": "str", "mistakes": {"0b678656a542a2d98995f0d2f3ffce79955b38449d4fe91f4a21d42b7a79d994": "plus on va profond, plus la reconstruction s'éloigne des pixels"}}
📝 Ex 28.Q8b : réponse enregistrée (str).
WB_ANSWER {"hash": "9f960359bb7c9d875216d6f559456592ab04643618372789e4bc4026aa2e05c7", "id": "28.Q8c", "kind": "str", "mistakes": {"394ba23579e6364406aaf7767c97ccae960beb309521456cd695f3e0f72d33d0": "toutes les couches, c'est le choix du livre pour le style", "a0a3b5657c2f4192762c86be2d975ceec982672caad2f3f99d2fc399e59e987b": "c'est le choix du code de Chollet, pas 

**28.Q9 — Loss de style, poids relatifs et variation totale**

In [11]:
wb.record("28.Q9a", "B", mistakes={"relis le §28.4.5 : le livre prend le style sur toutes les convolutions": "A"})
wb.record("28.Q9b", round(1 / 0.025), mistakes={"refais la division : 1 divisé par 0,025": 25})
wb.record("28.Q9c", "C", mistakes={"la variation totale compare chaque pixel à ses voisins, pas à une plage de couleurs": "A", "l'écart à la photo, c'est la loss de contenu": "B"})
wb.record("28.Q9d", True, mistakes={"multiplie la loss de style par 1 000 en changeant de normalisation : le même β donne-t-il le même dosage ?": False})
wb.record("28.Q9e", False, mistakes={"partir de la photo, c'est partir d'une image dont le contenu est déjà juste : c'est elle qui converge vite": True})

WB_ANSWER {"hash": "37e608d47ad2c1d21fa8cc2953e8d56966b16253dd4c6ca77df9e9c1d6b46da2", "id": "28.Q9a", "kind": "str", "mistakes": {"8bb06dcbddc6f80f9f4406792848a72de398f7a394ffc7e691b1db76517f6336": "relis le §28.4.5 : le livre prend le style sur toutes les convolutions"}}
📝 Ex 28.Q9a : réponse enregistrée (str).
WB_ANSWER {"hash": "fa351441fcea156e57c357e392ffb6f9fec4f15f05fe3edcb20153804670e288", "id": "28.Q9b", "kind": "int", "magnitude": 1, "mistakes": {"54c11942fcec319dcdd21c7adb86b50d50d7c30f2efcb107bf2ba48f3b5e795f": "refais la division : 1 divisé par 0,025"}, "sign": 1}
📝 Ex 28.Q9b : réponse enregistrée (int).
WB_ANSWER {"hash": "0cd2a93f9eccba478e85a2df719a840e943b7bf6c1ac6be9570c0b8680a4095c", "id": "28.Q9c", "kind": "str", "mistakes": {"5648b41a7f6b6a2a54d094aae3abcd47183e73ba90698444cb15702cd5344817": "la variation totale compare chaque pixel à ses voisins, pas à une plage de couleurs", "8f768471b330c4e16ac02064977ebf8853c257624266c019386fa686775c0bf5": "l'écart à la photo,

**28.Q10 — Générer du texte mot à mot : ce qu'implique « le mot le plus probable »**

In [12]:
wb.record("28.Q10a", "A", mistakes={"relis le §28.5 : « le mot suivant le plus probable »": "B"})
wb.record("28.Q10b", False, mistakes={"le glouton ne tire rien au hasard : même entrée, même sortie": True})
wb.record("28.Q10c", "B", mistakes={"un modèle mot à mot choisit dans son vocabulaire : il ne peut pas inventer de mot": "A"})
wb.record("28.Q10d", False, mistakes={"la couche de sortie a un neurone par mot du vocabulaire : rien d'autre ne peut sortir": True})
wb.record("28.Q10e", "A", mistakes={"diviser par T < 1, c'est multiplier les écarts entre logits": "B"})

WB_ANSWER {"hash": "0ac416a7410ca42f63a48bb0f4eff319674184d02058b06912b0b0ad19d35941", "id": "28.Q10a", "kind": "str", "mistakes": {"be6bf61f4dbe88916c53c98665de69e43fa15035c01bf698b1bd3e2c9ab01154": "relis le §28.5 : « le mot suivant le plus probable »"}}
📝 Ex 28.Q10a : réponse enregistrée (str).
WB_ANSWER {"hash": "64166dbb3ec22cea47544b2d78493e0dc6f545427b88640a3d494a82281bf72a", "id": "28.Q10b", "kind": "bool", "mistakes": {"68a651408b7e09fe566099f57e33e3788123d21e180ab6445dce7b74b036084d": "le glouton ne tire rien au hasard : même entrée, même sortie"}}
📝 Ex 28.Q10b : réponse enregistrée (bool).
WB_ANSWER {"hash": "c8bca6ac6abb57e031b5487c4b07f3bfcc5ffc35d0357993739adf61c5c2f8c1", "id": "28.Q10c", "kind": "str", "mistakes": {"7a68b25d6771d12ff709a1928b1b09fd1619662c4f4ac8ddc9e3c9881a1f9941": "un modèle mot à mot choisit dans son vocabulaire : il ne peut pas inventer de mot"}}
📝 Ex 28.Q10c : réponse enregistrée (str).
WB_ANSWER {"hash": "06ea8c35afb695d0805b0156a9f9e99836d7af6de5e4

**28.R1 — Un réseau qui sert de loss (le discriminateur)**

In [13]:
wb.record("28.R1a", "A", mistakes={"le générateur ne voit jamais de vraie image (ch. 27)": "B"})
wb.record("28.R1b", "B", mistakes={"VGG16 est gelé : il n'apprend rien pendant le transfert": "A"})
wb.record("28.R1c", True, mistakes={"dans les deux cas, le gradient de la loss traverse un réseau avant d'arriver à ce qu'on modifie": False})
wb.record("28.R1d", "B", mistakes={"chez Gatys, il n'y a pas de générateur : on modifie l'image elle-même": "A"})
wb.record("28.R1e", "B", mistakes={"une loss adversariale vient d'un discriminateur entraîné en même temps": "A"})

WB_ANSWER {"hash": "8b7942c8afd22f32753de39131dfa8598fde545cdc2ae777e909038be6969ea0", "id": "28.R1a", "kind": "str", "mistakes": {"7b60b0d819611c4e8a471b58d2a29a7ef8a76dbbe2a3fd39ea0eead82f48ce70": "le générateur ne voit jamais de vraie image (ch. 27)"}}
📝 Ex 28.R1a : réponse enregistrée (str).
WB_ANSWER {"hash": "8b16352111938ee1779ddc834e721ac0088ef2aa1d7b18bce0262a91a7f9dd3f", "id": "28.R1b", "kind": "str", "mistakes": {"5d4ab55898d284ef9fddbecbafc53b4c320db49244534f45698398b53adafaa2": "VGG16 est gelé : il n'apprend rien pendant le transfert"}}
📝 Ex 28.R1b : réponse enregistrée (str).
WB_ANSWER {"hash": "7db5aa9148a6bdd8f9508d6c614c3d120e8f1ba9eb31afbe9ec03a9939ef5327", "id": "28.R1c", "kind": "bool", "mistakes": {"84e56fa905c8629d1f00063189b60b776918f4153a52e381d0db7b516cc1fc68": "dans les deux cas, le gradient de la loss traverse un réseau avant d'arriver à ce qu'on modifie"}}
📝 Ex 28.R1c : réponse enregistrée (bool).
WB_ANSWER {"hash": "a5fe25838e991bf56c0f2c4ea823f2bf6adb819b6

**28.R2 — Mélange de contenu contre mélange de représentations**

In [14]:
wb.record("28.R2a", "B", mistakes={"une moyenne de pixels superpose les deux traits, chacun à moitié d'intensité": "A"})
wb.record("28.R2b", "A", mistakes={"le décodeur transforme un code en une image qui ressemble aux données": "B"})
wb.record("28.R2c", True, mistakes={"le transfert de style ne moyenne pas de pixels : il impose des activations et des matrices de Gram": False})
wb.record("28.R2d", round(0.75 * 0 + 0.25 * 4), mistakes={"t pondère z₂, et 1 − t pondère z₁": 3})
wb.record("28.R2e", False, mistakes={"la matrice de Gram est quadratique en F : développe (F₁ + F₂)(F₁ + F₂)ᵀ / 4": True})

WB_ANSWER {"hash": "990e7bf46daba7583732dfd3242662aaa43deaa38e7095030f730ce333055636", "id": "28.R2a", "kind": "str", "mistakes": {"badf33867f03f79e2ede5c8ffe46ef7d874891adadc415624d2506ad6d78c1f1": "une moyenne de pixels superpose les deux traits, chacun à moitié d'intensité"}}
📝 Ex 28.R2a : réponse enregistrée (str).
WB_ANSWER {"hash": "7dd537e9f94697b4cfc7acc872563b772a5ff5cfb90bcddaeb813d42029e740b", "id": "28.R2b", "kind": "str", "mistakes": {"3f5c5b2764564b5f788213364b37460aff6a4cce7a8d4b7a13b50410be9efdcf": "le décodeur transforme un code en une image qui ressemble aux données"}}
📝 Ex 28.R2b : réponse enregistrée (str).
WB_ANSWER {"hash": "8c495842011e003bdb1a943c060be4ed4e063a14663d6cab0a03b0e7d559b359", "id": "28.R2c", "kind": "bool", "mistakes": {"c9c0bac9fa84c15ca8afc57061facd91c104602af57ddde790376b776712ab83": "le transfert de style ne moyenne pas de pixels : il impose des activations et des matrices de Gram"}}
📝 Ex 28.R2c : réponse enregistrée (bool).
WB_ANSWER {"hash": "

**28.R3 — Champ récepteur et hiérarchie des filtres de VGG16**

In [15]:
wb.record("28.R3a", 3 + 2, mistakes={"chaque convolution 3 × 3 de plus ajoute 2 pixels au champ récepteur, pas 3": 6,
          "deux convolutions ne voient pas seulement 3 pixels : la seconde voit 3 sorties de la première": 3})
wb.record("28.R3b", 3 + 2 + 2, mistakes={"chaque convolution 3 × 3 de plus ajoute 2 pixels, pas 3": 9})
wb.record("28.R3c", True, mistakes={"relis le ch. 21 : chaque couche voit une plus grande région de l'image": False})
wb.record("28.R3d", 32 // 2 ** 5, mistakes={"il y a cinq poolings, pas quatre": 2,
          "seuls les poolings réduisent la taille, et il y en a cinq": 0})
wb.record("28.R3e", 64 * 64 * 9 + 64, mistakes={"n'oublie pas les 64 biais": 36864,
          "un filtre 3 × 3 voit tous les canaux d'entrée : 64 × 3 × 3 poids par filtre": 640})

WB_ANSWER {"hash": "4bb20f4033e2202a0ed1c3b49a9721ccd346b438a6e78a557ca3ca3e100f128b", "id": "28.R3a", "kind": "int", "magnitude": 0, "mistakes": {"2576edf6ad49fd4df3d4895bbbcd4c490f7276332af7eb42df20df7e71da2c2c": "deux convolutions ne voient pas seulement 3 pixels : la seconde voit 3 sorties de la première", "d263ec6f5ae3fbda8c16e445b6a1881e9561700bcac75cc6d685e293dd0a5ca5": "chaque convolution 3 × 3 de plus ajoute 2 pixels au champ récepteur, pas 3"}, "sign": 1}
📝 Ex 28.R3a : réponse enregistrée (int).
WB_ANSWER {"hash": "b579654d13b6db34fe083d52faa071cb89bffddd9eeebdd62c4e15ca275c04bc", "id": "28.R3b", "kind": "int", "magnitude": 0, "mistakes": {"d4c1ed3fff61b978ca3dc32473f97dc32157e70a752f3466007ecb782a044a8b": "chaque convolution 3 × 3 de plus ajoute 2 pixels, pas 3"}, "sign": 1}
📝 Ex 28.R3b : réponse enregistrée (int).
WB_ANSWER {"hash": "a72f84a13b23e9d951bde0a365da583a5142907112ce7a5d28a7199f13d2041e", "id": "28.R3c", "kind": "bool", "mistakes": {"e963914cc15a2f908ba9c13f6b53c

**Ex 28.1 — Tailles des activations et des matrices de Gram le long de VGG16**

In [16]:
wb.record("28.1a", 64 * side(1) ** 2, mistakes={"n'oublie pas les 64 canaux": 50176,
          "la convolution garde la taille : 224 × 224, pas 112 × 112": 64 * 112 ** 2})
wb.record("28.1b", 512 * side(4) ** 2, mistakes={"le bloc 4 vient après trois poolings : ses cartes font 28 × 28": 512 * 14 ** 2,
          "la première convolution du bloc 4 a déjà 512 filtres": 256 * 28 ** 2})
wb.record("28.1c", 512 ** 2, mistakes={"la matrice de Gram est C × C : elle ne dépend pas de la taille des cartes": 401408,
          "c'est la moitié de la matrice : compte tous ses coefficients": 512 * 513 // 2})
wb.record("28.1d", 64 * side(1) ** 2 // 64 ** 2, mistakes={"divise le nombre de valeurs de l'activation par 64 × 64, la taille de la matrice": 64 * 224 ** 2 // 64})
wb.record("28.1e", 512 * 513 // 2, mistakes={"la diagonale compte aussi : n(n + 1)/2, pas n(n − 1)/2": 512 * 511 // 2,
          "la matrice est symétrique : les coefficients de part et d'autre de la diagonale sont égaux": 512 ** 2})
wb.record("28.1f", side(5, image=128), mistakes={"le bloc 5 vient après quatre poolings, pas cinq": 4,
          "le bloc 5 vient après quatre poolings, et chacun divise le côté par 2": 16})
wb.record("28.1g", False, mistakes={"la matrice de Gram est C × C : seul le nombre de filtres compte": True})

WB_ANSWER {"hash": "0f40da9941a6b912c190bf5b035167bff8dd03bc8b6891f86bff698122ec2425", "id": "28.1a", "kind": "int", "magnitude": 6, "mistakes": {"867b55d0146ab3dce7804f400718d63c12e7cd0ca74817785f2fbe3e580f0f9e": "n'oublie pas les 64 canaux", "d52f53241186da389971bdae0f5f7a0ee6e64d395f04c365ecd5b17c4c74cb1e": "la convolution garde la taille : 224 × 224, pas 112 × 112"}, "sign": 1}
📝 Ex 28.1a : réponse enregistrée (int).
WB_ANSWER {"hash": "054820a336609ff366432bbb2bf2446a14dbf5c5e4c4728d2097b64a4e5c2f7b", "id": "28.1b", "kind": "int", "magnitude": 5, "mistakes": {"426549d6a787169986cfef3c0cc6d55342e7a56fa11e51a29f239fb12790d4cb": "la première convolution du bloc 4 a déjà 512 filtres", "46aee663a0a0c23476a81e191a874870ed7d9aac76b81d5418762d214ef0a16b": "le bloc 4 vient après trois poolings : ses cartes font 28 × 28"}, "sign": 1}
📝 Ex 28.1b : réponse enregistrée (int).
WB_ANSWER {"hash": "4ae18fcdb871ac4f39a29d94bf26e4916a75c4435b280d34a98117c944e7a095", "id": "28.1c", "kind": "int", "m

**Ex 28.2 — Une matrice de Gram à la main**

In [17]:
wb.record("28.2a", int(RAW_28_2[0, 1]), mistakes={"le coefficient (1, 2) combine deux lignes différentes de F, pas une ligne avec elle-même": 6})
wb.record("28.2b", int(RAW_28_2[1, 1]), mistakes={"le coefficient (2, 2) est la somme des carrés de la ligne 2, pas sa somme": 5})
wb.record("28.2c", G_28_2[0, 2], decimals=3, mistakes={"normalise par C·H·W = 3 × 2 × 2 = 12": 12.0,
          "normalise par C·H·W = 12, pas seulement par H·W = 4": 3.0})
wb.record("28.2d", G_28_2[1, 1], decimals=3, mistakes={"normalise par C·H·W = 12": 11.0,
          "normalise par C·H·W = 12, pas seulement par H·W = 4": 2.75})
wb.record("28.2e", G_28_2.sum(), decimals=3, mistakes={"compte chaque coefficient hors diagonale deux fois : la matrice en a neuf": 0.5 + 11 / 12 + 2 + 0.25 + 1 + 0.5,
          "normalise par C·H·W = 12": 83.0})
wb.record("28.2f", G_28_2[0, 1] / np.sqrt(G_28_2[0, 0] * G_28_2[1, 1]), decimals=3, mistakes={"divise par la racine du produit des deux diagonales, pas par leur produit": 3 / 66,
          "divise par la racine des deux coefficients diagonaux, pas par leur somme": 3 / 17})
wb.record("28.2g", True, mistakes={"retourner toutes les cartes de la même façon ne fait que réordonner les positions": False})

WB_ANSWER {"hash": "ea353f698aab10a19ea8ecdc6bcc04c69779302193fe5972e0219fb94604f783", "id": "28.2a", "kind": "int", "magnitude": 0, "mistakes": {"4fbceebc59a2285fe5abe130dc3ed20d25cf12419a1d5a0e1012304ea60536eb": "le coefficient (1, 2) combine deux lignes différentes de F, pas une ligne avec elle-même"}, "sign": 1}
📝 Ex 28.2a : réponse enregistrée (int).
WB_ANSWER {"hash": "65bad9b19f91f30dec146a421764b9d47d222420aa50cb5481936780981bc07f", "id": "28.2b", "kind": "int", "magnitude": 1, "mistakes": {"729cf1d21e8f8c6bd937041d3eef5c050b0eb3df5a8547a17acd985c85655fd1": "le coefficient (2, 2) est la somme des carrés de la ligne 2, pas sa somme"}, "sign": 1}
📝 Ex 28.2b : réponse enregistrée (int).
WB_ANSWER {"decimals": 3, "hash": "eb0ede69b41d75ce81775d45d41c77b6ec9ab19610384e8f7b83aa3eb74a876e", "hash_coarse": "d62cdc67781ed328e70158b69f1d47546775276a78cb0c57686df8e2fc32cc96", "id": "28.2c", "kind": "float", "magnitude": 0, "mistakes": {"28ffd83b2614775eb2f9318319acb47614401c6b0525d43e5b18

**Ex 28.4 — Losses de contenu, de style et de variation totale sur un mini-exemple**

In [18]:
wb.record("28.4a", LC_28_4, decimals=3, mistakes={"additionne les carrés des écarts, pas les écarts": 1.0,
          "additionne les carrés des écarts, pas leurs valeurs absolues": 3.0})
wb.record("28.4b", LC_28_4 / 4, decimals=3, mistakes={"F.mse_loss prend la moyenne des carrés : divise par 4": 5.0})
wb.record("28.4c", GX_28_4[0, 1], decimals=3, mistakes={"normalise par C·H·W = 2 × 2 = 4": 2.0,
          "G multiplie les lignes de F (une par filtre) : F Fᵀ, pas Fᵀ F": 0.25})
wb.record("28.4d", LS_28_4, decimals=3, mistakes={"additionne les carrés des quatre écarts, sans racine": np.sqrt(0.75),
          "n'oublie pas les deux coefficients hors diagonale, qui comptent chacun": 0.5})
wb.record("28.4e", LTV_28_4, decimals=3, mistakes={"additionne les deux directions, horizontale et verticale": 7.0,
          "élève chaque écart au carré : la valeur absolue ne suffit pas": 10.0,
          "des écarts signés se compensent : élève-les au carré": 6.0})
wb.record("28.4f", 0.025 * LC_28_4 + LS_28_4 + 1e-4 * LTV_28_4, decimals=4, mistakes={"le contenu est pondéré par α = 0,025": 5 + 0.75 + 14e-4,
          "n'oublie pas la variation totale, même petite": 0.875})
wb.record("28.4g", float(((SX_28_4 @ SX_28_4.T - SS_28_4 @ SS_28_4.T) ** 2).sum()), decimals=3, mistakes={"sans normalisation, les écarts sont 4 fois plus grands, leurs carrés 16 fois": 3.0})

WB_ANSWER {"decimals": 3, "hash": "6217e81df67e2261b817a74a6ab5855a0d4a9cc6fd8e4efaa2fc5955c0382b49", "hash_coarse": "5e446d57a6238ad83f8af1631673d7364c0cc5a62eae2a6b5b1f94f5c965273f", "id": "28.4a", "kind": "float", "magnitude": 0, "mistakes": {"94da5a866da9ff88d9acfc66997c0b9efa7299ec14d919886905aa3a03ecc241": "additionne les carrés des écarts, pas les écarts", "ab886a456d614761ad3c67f9a7e929fb2e862f620d356f8ecef5147da49a0b8a": "additionne les carrés des écarts, pas leurs valeurs absolues"}, "sign": 1}
📝 Ex 28.4a : réponse enregistrée (float, 3 décimale(s)).
WB_ANSWER {"decimals": 3, "hash": "ac321f32102aaadee99a536b9203efdbda022ffd022ce2fc5495a1632f87e299", "hash_coarse": "32fa6508616b245e517d970d2e27436c6b10ddd5779a9e1fd9ebb7be98bba480", "id": "28.4b", "kind": "float", "magnitude": 0, "mistakes": {"1b262142b02568288268906452e5af9b57cd1988277063b571675fc6d5169362": "F.mse_loss prend la moyenne des carrés : divise par 4"}, "sign": 1}
📝 Ex 28.4b : réponse enregistrée (float, 3 décimal

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu écrire une montée de gradient sur l'image d'un réseau gelé, normalisation et plage des pixels comprises ?
2. Sais-tu calculer une matrice de Gram et dire ce qu'elle garde et ce qu'elle oublie ?
3. Sais-tu régler les trois poids d'un transfert de style en lisant la valeur de chaque terme ?
4. Sais-tu choisir une stratégie de décodage, et dire ce qu'est la perplexité ?

**Pour aller plus loin** : l'article de Gatys et coll. (2015) et l'article « Feature Visualization » de Distill (2017), cités dans la fiche. La suite : le ch. 29, les datasets et la préparation du projet final.

*Les exercices de code 28.11 à 28.29 (parties A à D : la visualisation, le deep dream, le transfert de style et la génération de texte) seront ajoutés à la prochaine session de génération ; relance alors `python tools/start_chapter.py 28` pour obtenir le notebook complet.*